# IMPACT from Python — a guided tour with `push_circle`

This notebook walks through the whole Python surface of IMPACT using one task:
**planar disk pushing**. A pusher has to move a disk to a goal, and the scenario
is arranged so the obvious action is wrong — *the disk's goal is where the pusher
already stands*, so pushing from the initial contact drives the disk away from it.
The pusher has to travel around the disk and push from the far side.

Nothing in the code scripts that manoeuvre. It falls out of a complementarity
constraint, which is the point of the whole exercise.

We go from the top down:

1. [The one-liner](#1) — solve, report, save, draw
2. [What a task actually is](#2) — the model, in ~30 lines of CasADi
3. [The transcription](#3) — multiple vs single shooting
4. [The solver's own API](#4) — no tasks, no registry, just an MPCC
5. [Configuration](#5) — the 60 hyper-parameters and which ones matter
6. [Where the time goes](#6)
7. [Writing your own task](#7)

---

### Layout, so the imports make sense

| package | what it is | installed by `pip install .`? |
|---|---|---|
| `impact` | the solver: generic MPCC assembly, shooting transcriptions, the AuLa solve. **Knows no tasks.** | yes |
| `examples` | this repository's task models, tuned settings, drivers, visualizers | no — it is repository material |

So `impact` is what you depend on, and `examples` is what you read and copy.

In [ ]:
import sys, pathlib

# `examples` is not installed, so point Python at the checkout's python/ directory.
REPO = pathlib.Path.cwd()
while not (REPO / "impact_solver").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent

# append, NOT insert(0): python/ also holds the `impact` *source* tree, whose copy
# has no compiled extension. Ahead of site-packages it would shadow the installed
# solver and `import impact` would fail on `_impact_core`.
sys.path.append(str(REPO / "python"))

import numpy as np
import impact

print("impact", impact.__version__, "->", impact.__file__)
print("repo   ", REPO)

<a id="1"></a>
## 1. The one-liner

Every example is a directory, and `push_circle/task.py` is the whole of this
one's Python API: `config()` is the tuned configuration the paper's runs used,
and `solve()` hands back a `Result` that everything else hangs off. Nothing is
registered anywhere for this to work — the directory *is* the declaration.


In [ ]:
from examples.push_circle.task import config, solve

result = solve(config(horizon=100))
print(result.summary())


Read the last four lines: the pusher swept ~187° around the disk, never came
closer than the disk radius (non-penetration held at every knot), and first made
contact at step 13 — after it had already travelled most of the way around.

That is the local-minimum escape. Nobody told it to go around.

`Result` forwards every solver statistic, so you can also just ask:

In [ ]:
print(f"converged          {result.converged}")
print(f"objective          {result.objective_value:.6f}")
print(f"complementarity    {result.complementarity_violation:.3e}")
print(f"dynamics violation {result.dynamics_violation:.3e}")
print(f"outer iterations   {result.outer_iterations}")
print()
print("state trajectory  ", result.state.shape, " (nx, horizon+1)")
print("control trajectory", result.control.shape, " (nu, horizon)")


### Saving and drawing

`result.save()` writes the same trajectory text format the C++ drivers write,
into `results/<task>/<planner>/`. Drawing is the example's *other* file,
`push_circle/viz.py`, and it reads that saved file — which is why the animation
and the trajectory on disk can never disagree.


In [ ]:
path = result.save()
print("trajectory:", path.relative_to(REPO))
print("planner tag:", result.planner)   # names what ran; also the output subdirectory


```python
from examples.push_circle.viz import render

render(result.path)      # -> results/push_circle/push_circle.gif  (matplotlib + pillow)
```

Left commented out because rendering 100 frames takes a while. The same thing
from a shell:

```bash
python python/examples/run.py push_circle --horizon 100 --visualize
python python/examples/run.py push_circle --render-only       # newest saved trajectory
python python/examples/push_circle/main.py --visualize        # identical to the first
```


### Nothing is registered

`run.py` lists the directories next to it that contain a `main.py`; that is the
entire mechanism. An example's `main.py` is a thin argument parser over the same
`config()` and `solve()` used above, so the command line and the library cannot
drift apart — there is only one description of the task.


In [ ]:
import inspect
from examples import example_names, example_summary

for name in example_names():
    print(f"  {name:<16} {example_summary(name)}")

print("\npush_circle's knobs are just config()'s signature:")
print("  config", inspect.signature(config))


`config()` returns a plain `AulaConfig`, so any of the solver's 60
hyper-parameters is reachable by setting the field:


In [ ]:
cfg = config(horizon=60, angle=180.0)
cfg.rho_max = 400.0
tighter = solve(cfg)
print(f"angle 180 deg, rho_max 400 -> converged={tighter.converged} "
      f"goal_err={tighter.goal_error:.3e} outer={tighter.outer_iterations}")

# ...and a typo is an error, not a silently ignored setting.
try:
    impact.apply_config(cfg, rho_maks=400.0)
except AttributeError as error:
    print("\ntypo ->", error)


That matters more than it looks. A dropped `rho_max` still produces a perfectly
plausible-looking trajectory, so a silently ignored keyword is the worst possible
failure mode for this kind of code.

---
<a id="2"></a>
## 2. What a task actually is

`push_circle` is about thirty lines of CasADi. Here is the whole model:

```
state   x = [qx, qy, sx, sy]      disk centre q, pusher point s
control u = [f_n, f_t, vx, vy]    normal force, friction force, pusher velocity
```

with `d = s - q`, `r = ||d||`, and the smooth signed distance `phi(x) = r - R`.

In [ ]:
import inspect
from examples.push_circle.task import PushCircle

source = inspect.getsource(PushCircle)
print(source[source.index("    def dynamics"):])

The complementarity row is the entire trick:

$$0 \le f_n \;\perp\; \phi(x) \ge 0, \qquad \phi(x) = \lVert s - q\rVert - R$$

It does two jobs at once. `f_n = 0` unless `phi = 0` switches the contact force on
only when the pusher is touching; and `phi >= 0` is **non-penetration** — the
pusher cannot pass through the disk. A trajectory that reaches the goal therefore
*has* to route around it. That is where the go-around comes from.

The friction cone `|f_t| <= mu_c * f_n` is an ordinary stage inequality, and it
collapses to `f_t = 0` when there is no contact, for free.

### The cost

`push_circle` overrides exactly one thing from the generic quadratic cost: the
terminal row penalizes the *disk* only, leaving the pusher's final pose free.

In [ ]:
from examples.push_circle.task import PushCircleStage

print(inspect.getdoc(PushCircleStage))

---
<a id="3"></a>
## 3. The transcription

The task says nothing about how the horizon is laid out. That choice is made
outside it, by picking a shooting front-end:

* **multiple shooting** — `X` and `U` are both decision variables and the
  dynamics are defect equalities. Better conditioned; used for the paper's CITO
  results.
* **single shooting** — only `U` is free; the state is rolled out symbolically.
  Fewer variables, worse conditioning.

Both take the same stage description and return the same `ShootingSolution`.

In [ ]:
from examples.push_circle.task import PushCircle, PushCircleStage
from impact import MultipleShootingSolver, SingleShootingSolver

# Horizon 25, not 100: single shooting gets expensive fast here -- see below.
cfg = config(horizon=25, distance=1.5, angle=225.0)

for name, FrontEnd in [("multiple", MultipleShootingSolver),
                       ("single  ", SingleShootingSolver)]:
    s = FrontEnd(PushCircle(), stage_factory=PushCircleStage).solve(cfg)
    print(f"{name}  n_opt={s.z.size:4d}  converged={s.converged}  "
          f"obj={s.objective_value:.4f}  comp={s.complementarity_violation:.2e}  "
          f"time={s.solve_time:.3f}s")


Same objective to four decimals — they describe the same problem — from less than
half the variables, and it still takes ~50x longer.

That is the trade, and it gets worse with the horizon. Single shooting rolls the
state out symbolically, so knot `k` depends on *every* control before it: the
Jacobian fills in, and both evaluation and factorization stop being sparse.
Measured on this task:

| horizon | multiple `n_opt` / time | single `n_opt` / time |
|---|---|---|
| 15 | 124 / 0.005 s | 60 / 0.001 s |
| 25 | 204 / 0.21 s | 100 / 9.8 s |
| 40 | 324 / 0.24 s | 160 / 65 s |
| 60 | 484 / 0.33 s | 240 / 102 s |

Multiple shooting is essentially flat; single shooting is not. This is why the
paper's CITO results use multiple shooting, and why single shooting is kept for
comparison rather than for use.

The task's own `solve()` takes it as a keyword, and the command line as a flag:

```python
solve(config(horizon=25), mode="single")     # python
```
```bash
python python/examples/run.py push_circle --mode single
```

---
<a id="4"></a>
## 4. The solver's own API

Everything above is `examples`. Underneath, `impact` has no idea any of it
exists. Its actual interface is: *describe an MPCC, get a subproblem, solve it.*

$$\min_z \lVert \mathrm{cost}(z)\rVert^2 \quad\text{s.t.}\quad
\text{equality / inequality / complementarity blocks}$$

Here is a complete problem with no task, no horizon and no shooting:

In [ ]:
import casadi as ca
from impact import AulaConfig, BlockOptions, MPCCDescription, Solver, build_mpcc

target = np.array([0.8, 0.2, 0.25, 0.75])
z = ca.SX.sym("z", 4)

desc = MPCCDescription(z=z,
                       cost=z - ca.DM(target),   # objective is ||cost||^2
                       cost_is_linear=True)      # affine residual => quadratic objective
desc.add_complementarity("axis_a", z[0], z[1], BlockOptions(scale=1.0,  rho_init=1.0, tol=1e-8))
desc.add_complementarity("axis_b", z[2], z[3], BlockOptions(scale=0.75, rho_init=2.0, tol=1e-8))

cfg = AulaConfig()
cfg.max_outer_iters, cfg.rho_scale, cfg.rho_max = 300, 1.5, 1e3
cfg.outer_tol_h = cfg.outer_tol_comp = 1e-7
cfg.print_level = 0

r = Solver().solve(build_mpcc(desc).subproblem, cfg, np.zeros(4))
print("z* =", np.round(r.z, 10))
print(f"objective {r.objective_value:.10f}  (0.2^2 + 0.25^2 = 0.1025)")

The objective wants `z2 = 0.2` and `z3 = 0.25`, but complementarity forbids both
legs of a pair being positive, so one of each pair is driven to zero. Started from
`(0,0,0,0)` — the biactive corner, where every pair sits exactly on the kink and
neither branch is preferred.

Note the **two separate** complementarity blocks rather than one stacked block.
Each carries its own slacks, multipliers, penalty, conditioning scale and
tolerance, which is what you want when different groups have different natural
magnitudes.

Use this path when your problem is not a trajectory at all: an inverse problem, a
static contact configuration, a bilevel program in complementarity form.

### How the two sides actually talk

Python derives the augmented-Lagrangian residual and its Jacobian as CasADi
functions, and hands them to C++ in
CasADi's own **serialized** form. No CasADi expression is ever built in C++, and
no CasADi type appears in any binding signature.

The extension links the `libcasadi` that ships inside the `casadi` wheel, so both
sides of that boundary are the same library build by construction rather than by
version negotiation. `python/tests/test_parity.py` pins it: for every task, both
transcriptions, the Python-built residual and Jacobian are
compared against the C++ builders' at random points and must agree **to the last
bit**.

In [ ]:
built = build_mpcc(desc)
print("n_opt   ", built.n_opt, "  decision variables")
print("n_params", built.n_params, "  AuLa parameter buffer (penalties, multipliers, slacks)")
print("off_p   ", built.off_p, "  where *your* runtime parameters start")
print()
print("residual:", built.residual)

`p` — the runtime parameter block at `off_p` — is what lets the *same* symbolic
problem be re-solved with new data without rebuilding anything. That is exactly
how the Allegro MPC example runs at a control rate: the contact Jacobian and the
gap function are parameters, not structure.

---
<a id="5"></a>
## 5. Configuration

`AulaConfig` is the C++ struct itself, exposed field for field, so the Python
defaults *are* the solver's defaults and cannot drift.

In [ ]:
fields = impact.field_names()
print(len(fields), "hyper-parameters\n")
for i in range(0, len(fields), 4):
    print("  " + "".join(f"{f:<32}" for f in fields[i:i+4]))

You will not touch most of them. The ones that decide whether a hard problem
converges are, roughly:

| field | what it does |
|---|---|
| `rho_*_init`, `rho_max`, `rho_scale` | the penalty schedule per constraint channel |
| `*_scale` | per-channel conditioning; **the tuning that matters most here** |
| `outer_tol_h`, `outer_tol_g`, `outer_tol_comp` | feasibility targets |
| `max_inner_iters`, `inner_tol_init/final` | how hard to solve each subproblem |

The conditioning scales are not cosmetic — they were arrived at by measurement,
and an automatic-scaling replacement was tried and rejected. Each example's
`task.py` holds its own tuning, copied from the C++ drivers.

In [ ]:
from impact import config_to_dict

tuned = config_to_dict(config(horizon=100))
plain = config_to_dict(impact.AulaConfig())
print("push_circle's preset differs from the library defaults in:\n")
for key in sorted(tuned):
    if isinstance(tuned[key], list) or tuned[key] == plain[key]:
        continue
    print(f"  {key:<28} {plain[key]!r:>12}  ->  {tuned[key]!r}")

`config_to_dict` round-trips through JSON, which is the honest way to record what
a run actually used:

```python
import json
json.dump(impact.config_to_dict(config), open("run.json", "w"))
config = impact.config_from_dict(json.load(open("run.json")))
```

### One measurement trap worth knowing

The reported stationarity is *the same quantity the inner Gauss-Newton solver
stops on*. Asking for a 1e-8 certificate while leaving `newton_tol` at its default
makes **both** inner solvers appear to floor near that default — for reasons that
have nothing to do with either algorithm. `tighten_to_stationarity` moves them
together:

In [ ]:
c = impact.tighten_to_stationarity(impact.AulaConfig(), 1e-8)
print(f"stationarity_tol {c.stationarity_tol:.0e}   newton_tol {c.newton_tol:.0e}   "
      f"inner_tol_final {c.inner_tol_final:.0e}")

---
<a id="6"></a>
## 6. Where the solve time goes

`AulaResult` splits `solve_time` into CasADi evaluation and sparse factorization,
so "would a faster linear-algebra backend help?" is a measurement rather than a
guess.

In [ ]:
s = solve(config(horizon=100))
other = s.solve_time - s.eval_time - s.factor_time
print(f"solve  {s.solve_time*1000:7.1f} ms")
print(f"  eval   {s.eval_time*1000:7.1f} ms  {100*s.eval_time/s.solve_time:4.0f}%   CasADi residual/Jacobian")
print(f"  factor {s.factor_time*1000:7.1f} ms  {100*s.factor_time/s.solve_time:4.0f}%   sparse LDL^T + triangular solves")
print(f"  other  {other*1000:7.1f} ms  {100*other/s.solve_time:4.0f}%   assembly, projection, outer bookkeeping")

The two are co-dominant, with evaluation consistently the larger. Two
consequences, both already measured in the top-level README:

* a faster factorization is capped at 19–37% of total time, so doubling its speed
  buys 10–18% overall — for a new system dependency (CHOLMOD, MKL PARDISO, MUMPS);
* `config.jit = True` makes evaluation itself ~1.45× faster and keeps results
  bit-identical, but raises *total* solve time 3.4×, because per-call dispatch
  overhead swamps the small, frequently called functions. It is off by default.

---
<a id="7"></a>
## 7. Writing your own task

Subclass `impact.MPCCProblem` for an explicit-ODE task. Six methods, all
returning CasADi expressions:

In [ ]:
class RestingMass(impact.MPCCProblem):
    """A unit mass resting on the ground under gravity.

    x = [height, velocity], u = [contact force].
    0 <= f (perp) height >= 0  switches the force on only in contact *and*, through
    height >= 0, keeps the mass out of the ground -- the same two-jobs-at-once
    trick as push_circle's f_n (perp) phi.

    Nobody tells it the contact force. It has to come out of the solve.
    """

    state_dim   = property(lambda self: 2)
    control_dim = property(lambda self: 1)
    comp_dim    = property(lambda self: 1)     # one complementarity pair per stage
    time_step   = property(lambda self: 0.05)

    def dynamics(self, x, u):
        return ca.vertcat(x[1], u[0] - 1.0)    # gravity pulls toward the ground

    def G(self, x, u):
        return ca.vertcat(u[0])                # contact force

    def H(self, x, u):
        return ca.vertcat(x[0])                # height above the ground


cfg = impact.AulaConfig()
cfg.horizon = 30
cfg.x_0, cfg.x_goal = np.array([0.0, 0.0]), np.array([0.0, 0.0])
cfg.final_cost_weight, cfg.stage_cost_weight, cfg.stage_state_cost_weight = 100.0, 1e-3, 1.0
cfg.max_outer_iters = 600
cfg.outer_tol_h = cfg.outer_tol_g = 1e-5
cfg.outer_tol_comp = 1e-4
cfg.comp_scale, cfg.inner_tol_final = 10.0, 1e-5   # the two knobs every real task tunes
cfg.print_level = 0

sol = impact.MultipleShootingSolver(RestingMass()).solve(cfg)
height, force = sol.state_trajectory[0], sol.control_trajectory[0]
print("converged      ", sol.converged)
print("min height     ", f"{height.min():+.2e}   (>= 0: never went through the ground)")
print("contact force  ", f"[{force.min():.4f}, {force.max():.4f}]   (gravity is 1.0)")
print("complementarity", f"{sol.complementarity_violation:.2e}")

The contact force comes out within a few percent of **1.0** — cancelling gravity —
and the mass never penetrates the ground. Neither of those was specified; both
fall out of the complementarity row.

> **An honest caveat.** Even here the two lines above `max_outer_iters` matter:
> at library defaults this problem stalls above its complementarity target. It
> works at all because the contact is *persistent*.
> Change it to free flight followed by an impact and it stops converging at these
> tolerances: an impulsive contact under explicit Euler needs a real penalty
> schedule and conditioning scales, which is exactly what each task's `config()`
> exists to hold, and why `push_circle` is tuned rather than run at library
> defaults. Do not read a converging toy as evidence that a new contact task will
> converge untuned.

That is the whole interface. `eq`/`ineq` and their `*_dim` properties are
optional, and default to empty.

For a contact task whose dynamics are algebraic rather than an explicit ODE —
an LCP force balance — subclass `impact.LCPProblem` instead and use
`impact.LCPSingleShootingSolver`; `examples/allegro/task.py` is a full worked
example, including the receding-horizon MuJoCo loop.

To get it on the command line, put it in a directory:

```
my_task/
  task.py     the model, its config() and its solve()
  viz.py      how to draw a saved trajectory of it
  main.py     a docstring summary, and main(argv=None) -> int
```

`python python/examples/run.py list` will show it and `run.py my_task` will run
it, without being told it exists. `examples/common/cli.py` supplies the shared
flags and the print/save/draw tail, so `main.py` is usually about thirty lines.

---

## Cheat sheet

```python
from examples.push_circle.task import config, solve
from examples.push_circle.viz import render

cfg = config(horizon=100); cfg.rho_max = 400.0
result = solve(cfg)
result.summary(); result.result_line()
result.state; result.control; result.converged; result.solve_time
render(result.save())
```

```bash
python python/examples/run.py list                     # every example and its summary
python python/examples/run.py push_circle --visualize
python python/examples/push_circle/main.py --visualize # the same run, no dispatcher
python python/examples/run.py push_t --set rho_scale=1.25
python python/examples/run.py push_circle --print-config    # the tuned settings, as JSON
python python/examples/run.py push_circle --render-only     # newest saved trajectory
```

```python
import impact                                   # the solver alone; no tasks
impact.build_mpcc(impact.MPCCDescription(...))  # generic MPCC
impact.MultipleShootingSolver(MyTask()).solve(config)
impact.SingleShootingSolver(MyTask()).solve(config)
impact.LCPSingleShootingSolver(MyContactTask()).solve(config, q0, phi, J, ...)
```